### Import Libraries

In [4]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.1/12.8 MB 812.7 kB/s eta 0:00:16
     --------------------------------------- 0.1/12.8 MB 812.7 kB/s eta 0:00:16
     --------------------------------------- 0.1/12.8 MB 812.7 kB/s eta 0:00:16
     --------------------------------------- 0.1/12.8 MB 812.7 kB/s eta 0:00:16
     --------------------------------------- 0.1/12.8 MB 812.7 kB/s eta 0:00:16
     --------------------------------------- 0.1/12.8 MB 261.7 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 261.7 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 261.7 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 261.7 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 261.7 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB

ERROR: Exception:
Traceback (most recent call last):
  File "e:\MLP\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "e:\MLP\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "e:\MLP\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "e:\MLP\.venv\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 90, in read
    data = self.__fp.read(amt)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\http\client.py", line 466, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^

In [29]:
import os
import re
import pandas as pd
import numpy as np
import nltk
import spacy

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

### Download NLTK resources

In [6]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Load spaCy model

In [7]:
# Load spaCy small English model (no trailing space in model name)
nlp = spacy.load("en_core_web_sm")

### Create folder structure for data

In [8]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

### Download Dataset

In [9]:
from pathlib import Path
from urllib.request import urlretrieve

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

train_url = "https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/train/0000.parquet"
test_url = "https://huggingface.co/datasets/stanfordnlp/sentiment140/resolve/refs%2Fconvert%2Fparquet/sentiment140/test/0000.parquet"

train_path = raw_dir / "train.parquet"
test_path = raw_dir / "test.parquet"

urlretrieve(train_url, train_path)
urlretrieve(test_url, test_path)

print(f"Saved: {train_path}")
print(f"Saved: {test_path}")

Saved: data\raw\train.parquet
Saved: data\raw\test.parquet


### Load the datasets

In [30]:
train_df = pd.read_parquet(r"E:\MLP\Algorithms\supervised\NLP_pipeline\data\raw\train.parquet")
test_df = pd.read_parquet(r"E:\MLP\Algorithms\supervised\NLP_pipeline\data\raw\test.parquet") 
train_df.head()  

,text,date,user,sentiment,query
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",Mon Apr 06 22:19:45 PDT 2009,_TheSpecialOne_,0,NO_QUERY
1,is upset that he can't update his Facebook by ...,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,0,NO_QUERY
2,@Kenichan I dived many times for the ball. Man...,Mon Apr 06 22:19:53 PDT 2009,mattycus,0,NO_QUERY
3,my whole body feels itchy and like its on fire,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,0,NO_QUERY
4,"@nationwideclass no, it's not behaving at all....",Mon Apr 06 22:19:57 PDT 2009,Karoli,0,NO_QUERY


### Data Understanding

In [11]:
print("Shape:", train_df.shape)
print("Columns:", train_df.columns)
train_df.head(5)

Shape: (1600000, 5)
Columns: Index(['text', 'date', 'user', 'sentiment', 'query'], dtype='str')


,text,date,user,sentiment,query
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",Mon Apr 06 22:19:45 PDT 2009,_TheSpecialOne_,0,NO_QUERY
1,is upset that he can't update his Facebook by ...,Mon Apr 06 22:19:49 PDT 2009,scotthamilton,0,NO_QUERY
2,@Kenichan I dived many times for the ball. Man...,Mon Apr 06 22:19:53 PDT 2009,mattycus,0,NO_QUERY
3,my whole body feels itchy and like its on fire,Mon Apr 06 22:19:57 PDT 2009,ElleCTF,0,NO_QUERY
4,"@nationwideclass no, it's not behaving at all....",Mon Apr 06 22:19:57 PDT 2009,Karoli,0,NO_QUERY


### Select Required Columns

In [12]:
text_data = train_df[["text", "sentiment"]].dropna()
text_data.head()

,text,sentiment
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,is upset that he can't update his Facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,0
3,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",0


### Check Sentiment distribution

In [13]:
# Check label distribution in raw and working data
print('Raw train_df sentiment counts:')
print(train_df['sentiment'].value_counts(dropna=False).head())

print('\nWorking text_data sentiment counts:')
print(text_data['sentiment'].value_counts(dropna=False))

Raw train_df sentiment counts:
sentiment
0    800000
4    800000
Name: count, dtype: int64

Working text_data sentiment counts:
sentiment
0    800000
4    800000
Name: count, dtype: int64


### Convert sentiment to binary

In [ ]:
  # Rebuild from raw columns so reruns don't inherit previously filtered data
text_data = train_df[['text', 'sentiment']].dropna().copy()

# Robust binary sentiment conversion: keep 0 and map 4 -> 1
sentiment_num = pd.to_numeric(text_data['sentiment'], errors='coerce')
sentiment_num = sentiment_num.replace({4: 1})

mask = sentiment_num.isin([0, 1])
text_data = text_data.loc[mask].copy()
text_data['sentiment'] = sentiment_num.loc[mask].astype(int)

print(text_data['sentiment'].value_counts(dropna=False))
text_data.head()

sentiment
0    800000
1    800000
Name: count, dtype: int64


,text,sentiment
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,is upset that he can't update his Facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,0
3,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",0


### Text Cleaning function 

In [32]:

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_data['cleaned_text'] = text_data['text'].apply(clean_text)
text_data.head()

,text,sentiment,cleaned_text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,awww thats a bummer you shoulda got david carr...
1,is upset that he can't update his Facebook by ...,0,is upset that he cant update his facebook by t...
2,@Kenichan I dived many times for the ball. Man...,0,i dived many times for the ball managed to sav...
3,my whole body feels itchy and like its on fire,0,my whole body feels itchy and like its on fire
4,"@nationwideclass no, it's not behaving at all....",0,no its not behaving at all im mad why am i her...


### Saved Cleaned Data

In [16]:
text_data.to_csv("data/cleaned/cleaned_text.csv", index=False)

### Tokenization

In [17]:
sample = text_data['cleaned_text'].iloc[0]
sentences = sent_tokenize(sample)
word = word_tokenize(sample)
print("sentences:", sentences)
print("words:", word)


sentences: ['awww thats a bummer you shoulda got david carr of third day to do it d']
words: ['awww', 'thats', 'a', 'bummer', 'you', 'shoulda', 'got', 'david', 'carr', 'of', 'third', 'day', 'to', 'do', 'it', 'd']


### Stopword removal

In [33]:
stop_words = set(stopwords.words('english'))

filtered_words = [w for w in word if w not in stop_words]
filtered_words  

['awww', 'thats', 'bummer', 'shoulda', 'got', 'david', 'carr', 'third', 'day']

### Lemmatization

In [34]:
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_words]
lemmatized_words

['awww', 'thats', 'bummer', 'shoulda', 'got', 'david', 'carr', 'third', 'day']

### full preprocessing pipeline

In [35]:
def prepocess_pipeline(text):
    text = clean_text(text)
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)
text_data['preprocessed_text'] = text_data['cleaned_text'].apply(prepocess_pipeline)
text_data.head()

,text,sentiment,cleaned_text,preprocessed_text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,awww thats a bummer you shoulda got david carr...,awww thats bummer shoulda got david carr third...
1,is upset that he can't update his Facebook by ...,0,is upset that he cant update his facebook by t...,upset cant update facebook texting might cry r...
2,@Kenichan I dived many times for the ball. Man...,0,i dived many times for the ball managed to sav...,dived many time ball managed save rest go bound
3,my whole body feels itchy and like its on fire,0,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,"@nationwideclass no, it's not behaving at all....",0,no its not behaving at all im mad why am i her...,behaving im mad cant see


### Saved processed data

In [21]:
text_data.to_csv("data/processed/processed_text.csv", index=False)

### Named entity recognation

In [22]:
sample_text = text_data['preprocessed_text'].iloc[0]
doc = nlp(sample_text)

for ent in doc.ents:
    print(ent.text, ent.label_)

awww ORG
s bummer PERSON
david carr PERSON
third day DATE


### POS Tagging

In [23]:
for tocken in doc:
    print(tocken.text, tocken.pos_)


awww PROPN
that PRON
s VERB
bummer NOUN
shoulda PROPN
got VERB
david PROPN
carr PROPN
third ADJ
day NOUN


### Chunking

In [24]:
for chunk in doc.noun_chunks:
    print(chunk.text)

that
bummer shoulda
david carr


### Dependancy Parsing

In [25]:
for token in doc:
    print(token.text, token.dep_, token.head.text)

awww npadvmod s
that nsubj s
s ccomp got
bummer compound shoulda
shoulda nsubj got
got ROOT got
david compound carr
carr dobj got
third amod day
day npadvmod got


### Tf-IDF vectorization

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

# Keep only rows with valid label/text for modeling
model_data = text_data.dropna(subset=['preprocessed_text', 'sentiment']).copy()
model_data['sentiment'] = model_data['sentiment'].astype(int)

X = vectorizer.fit_transform(model_data['preprocessed_text'])
y = model_data['sentiment']

### Train test split

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train class counts:')
print(y_train.value_counts())
print('Test class counts:')
print(y_test.value_counts())

Train class counts:
sentiment
1    640000
0    640000
Name: count, dtype: int64
Test class counts:
sentiment
0    160000
1    160000
Name: count, dtype: int64


### Train model (Logistic regression)

In [38]:
from sklearn.linear_model import LogisticRegression

print('Classes in y_train:', sorted(y_train.unique().tolist()))

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

Classes in y_train: [0, 1]


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### Model evaluation

In [39]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.774275
              precision    recall  f1-score   support

           0       0.79      0.75      0.77    160000
           1       0.76      0.80      0.78    160000

    accuracy                           0.77    320000
   macro avg       0.77      0.77      0.77    320000
weighted avg       0.77      0.77      0.77    320000

